In [1]:
import pandas as pd
import json

#### Kiểm tra cấu trúc file football-data

In [2]:
# Mở file json
with open("data_epl/football_data_raw.json", "r", encoding ='utf-8') as file:
    raw_data = json.load(file)

matches = raw_data.get('matches',[])
print(f'Tổng số trận đấu: {len(matches)}')

Tổng số trận đấu: 380


In [3]:
# Lấy 1 trận để kiểm tra cấu trúc
first_match = matches[0]
print(json.dumps(first_match, indent=4))

{
    "area": {
        "id": 2072,
        "name": "England",
        "code": "ENG",
        "flag": "https://crests.football-data.org/770.svg"
    },
    "competition": {
        "id": 2021,
        "name": "Premier League",
        "code": "PL",
        "type": "LEAGUE",
        "emblem": "https://crests.football-data.org/PL.png"
    },
    "season": {
        "id": 2287,
        "startDate": "2024-08-16",
        "endDate": "2025-05-25",
        "currentMatchday": 38,
        "winner": null
    },
    "id": 497410,
    "utcDate": "2024-08-16T19:00:00Z",
    "status": "FINISHED",
    "matchday": 1,
    "stage": "REGULAR_SEASON",
    "group": null,
    "lastUpdated": "2025-06-08T20:20:25Z",
    "homeTeam": {
        "id": 66,
        "name": "Manchester United FC",
        "shortName": "Man United",
        "tla": "MUN",
        "crest": "https://crests.football-data.org/66.png"
    },
    "awayTeam": {
        "id": 63,
        "name": "Fulham FC",
        "shortName": "Fulham",
   

#### Kiểm tra cấu trúc file api-football

In [4]:
with open("data_epl/api_football_raw.json", "r", encoding='utf-8') as file:
    raw_api_football_data = json.load(file)

# Sửa 'fixtures' thành 'response' theo đúng chuẩn API-Football
fixtures = raw_api_football_data.get('response', [])
print(f'Tổng số trận đấu: {len(fixtures)}')


Tổng số trận đấu: 380


In [5]:
# Lấy 1 trận để kiểm tra cấu trúc
first_match = fixtures[0]
print(json.dumps(first_match, indent=4))

{
    "fixture": {
        "id": 1208021,
        "referee": "R. Jones",
        "timezone": "UTC",
        "date": "2024-08-16T19:00:00+00:00",
        "timestamp": 1723834800,
        "periods": {
            "first": 1723834800,
            "second": 1723838400
        },
        "venue": {
            "id": 556,
            "name": "Old Trafford",
            "city": "Manchester"
        },
        "status": {
            "long": "Match Finished",
            "short": "FT",
            "elapsed": 90,
            "extra": null
        }
    },
    "league": {
        "id": 39,
        "name": "Premier League",
        "country": "England",
        "logo": "https://media.api-sports.io/football/leagues/39.png",
        "flag": "https://media.api-sports.io/flags/gb-eng.svg",
        "season": 2024,
        "round": "Regular Season - 1",
        "standings": true
    },
    "teams": {
        "home": {
            "id": 33,
            "name": "Manchester United",
            "logo": "h

#### Làm phẳng dữ liệu

In [6]:
# ==========================================
# BƯỚC 1: LÀM PHẲNG DỮ LIỆU CƠ BẢN (BASE MATCHES)
# ==========================================
with open("data_epl/api_football_raw.json", "r", encoding="utf-8") as f:
    raw_base_data = json.load(f)

base_matches = raw_base_data.get("response", [])
base_list = []

for match in base_matches:
    fixture = match.get("fixture", {})
    teams = match.get("teams", {})
    goals = match.get("goals", {})
    score = match.get("score", {})

    base_list.append({
        "match_id": fixture.get("id"),
        "match_date": fixture.get("date"),
        "stadium": fixture.get("venue", {}).get("name"),
        "referee": fixture.get("referee"),
        "home_team": teams.get("home", {}).get("name"),
        "away_team": teams.get("away", {}).get("name"),
        "home_score": goals.get("home"),
        "away_score": goals.get("away"),
        "ht_home_score": score.get("halftime", {}).get("home"),
        "ht_away_score": score.get("halftime", {}).get("away")
    })

df_base = pd.DataFrame(base_list)
df_base['match_date'] = pd.to_datetime(df_base['match_date'])


# ==========================================
# BƯỚC 2: LÀM PHẲNG DỮ LIỆU THỐNG KÊ NÂNG CAO
# ==========================================
with open("data_epl/api_football_statistics.json", "r", encoding="utf-8") as f:
    raw_stats_data = json.load(f)

flattened_stats = []

for match in raw_stats_data:
    match_id = match.get("fixture_id")
    match_stats = match.get("statistics", [])

    if len(match_stats) < 2:
        continue

    home_data = match_stats[0].get("statistics", [])
    away_data = match_stats[1].get("statistics", [])

    row = {"match_id": match_id}

    # Đảo dòng thành cột cho Đội nhà
    for stat in home_data:
        stat_type = str(stat["type"]).replace(" ", "_").lower()
        row[f"home_{stat_type}"] = stat["value"]

    # Đảo dòng thành cột cho Đội khách
    for stat in away_data:
        stat_type = str(stat["type"]).replace(" ", "_").lower()
        row[f"away_{stat_type}"] = stat["value"]

    flattened_stats.append(row)

df_stats = pd.DataFrame(flattened_stats)


# ==========================================
# BƯỚC 3: KẾT HỢP HAI NGUỒN DỮ LIỆU (INNER JOIN)
# ==========================================
df_api_complete = pd.merge(df_base, df_stats, on="match_id", how="inner")

# Thống kê kết quả kiểm tra chất lượng dữ liệu
print("\n--- KẾT QUẢ NGHIỆM THU ĐƯỜNG ỐNG ---")
print(f"Số lượng trận ở bảng cơ bản : {df_base.shape[0]} hàng, {df_base.shape[1]} cột")
print(f"Số lượng trận ở bảng thống kê: {df_stats.shape[0]} hàng, {df_stats.shape[1]} cột")
print(f"Siêu bảng hoàn chỉnh sau gộp : {df_api_complete.shape[0]} hàng, {df_api_complete.shape[1]} cột")

# Hiển thị thử dữ liệu
df_api_complete.head()


--- KẾT QUẢ NGHIỆM THU ĐƯỜNG ỐNG ---
Số lượng trận ở bảng cơ bản : 380 hàng, 10 cột
Số lượng trận ở bảng thống kê: 380 hàng, 37 cột
Siêu bảng hoàn chỉnh sau gộp : 380 hàng, 46 cột


,match_id,match_date,stadium,referee,home_team,away_team,home_score,away_score,ht_home_score,ht_away_score,...,away_offsides,away_ball_possession,away_yellow_cards,away_red_cards,away_goalkeeper_saves,away_total_passes,away_passes_accurate,away_passes_%,away_expected_goals,away_goals_prevented
0,1208021,2024-08-16 19:00:00+00:00,Old Trafford,R. Jones,Manchester United,Fulham,1,0,0,0,...,1.0,45%,3.0,NaN,4.0,384,306,80%,0.44,1.07
1,1208022,2024-08-17 11:30:00+00:00,Portman Road,T. Robinson,Ipswich,Liverpool,0,2,0,0,...,0.0,62%,1.0,NaN,2.0,570,492,86%,2.65,0.24
2,1208025,2024-08-17 14:00:00+00:00,St. James' Park,C. Pawson,Newcastle,Southampton,1,0,1,0,...,2.0,78%,4.0,0.0,0.0,649,579,89%,1.77,-0.37
3,1208023,2024-08-17 14:00:00+00:00,Emirates Stadium,J. Gillett,Arsenal,Wolves,2,0,1,0,...,1.0,47%,2.0,NaN,4.0,375,307,82%,0.47,-0.30
4,1208024,2024-08-17 14:00:00+00:00,Goodison Park,S. Hooper,Everton,Brighton,0,3,0,1,...,1.0,62%,1.0,0.0,1.0,575,492,86%,1.43,0.05


#### Chuẩn hóa kiểu dữ li

In [7]:
# Tạo bản sao để xử lý
df_clean = df_api_complete.copy()

# Phân loại các nhóm cột cần xử lý đặc thù
float_columns = ['home_expected_goals', 'away_expected_goals', 'home_goals_prevented', 'away_goals_prevented']
pct_columns = ['home_ball_possession', 'away_ball_possession', 'home_passes_%', 'away_passes_%']
exclude_from_int = ['match_id', 'match_date', 'stadium', 'referee', 'home_team', 'away_team'] + float_columns + pct_columns

# 1. ÉP KIỂU SỐ NGUYÊN (INT) CHO CÁC CỘT ĐẾM (Thẻ phạt, Việt vị, Sút...)
# Lấy tất cả các cột còn lại không nằm trong danh sách loại trừ ở trên
int_columns = [col for col in df_clean.columns if col not in exclude_from_int]

for col in int_columns:
    # Biến NaN thành 0 (Không có thống kê = 0 lần), sau đó ép về int
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0).astype(int)

# 2. XỬ LÝ CỘT PHẦN TRĂM (Ví dụ: "55%" -> 55)
for col in pct_columns:
    if col in df_clean.columns:
        # Xóa ký tự '%', ép về số nguyên
        df_clean[col] = df_clean[col].astype(str).str.replace('%', '', regex=False)
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0).astype(int)

# 3. ÉP KIỂU SỐ THỰC (FLOAT) CHO XG VÀ BÀN THẮNG CỨU THUA
for col in float_columns:
    if col in df_clean.columns:
        # Đảm bảo xG luôn là số thập phân
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0.0).astype(float)

print("Đã chuẩn hóa xong! Kiểm tra các cột mẫu:")
print(df_clean[['home_red_cards', 'away_offsides', 'home_ball_possession', 'home_expected_goals']].dtypes)

# Hiển thị thử dữ liệu đã làm sạch
df_clean.head()

Đã chuẩn hóa xong! Kiểm tra các cột mẫu:
home_red_cards            int64
away_offsides             int64
home_ball_possession      int64
home_expected_goals     float64
dtype: object


,match_id,match_date,stadium,referee,home_team,away_team,home_score,away_score,ht_home_score,ht_away_score,...,away_offsides,away_ball_possession,away_yellow_cards,away_red_cards,away_goalkeeper_saves,away_total_passes,away_passes_accurate,away_passes_%,away_expected_goals,away_goals_prevented
0,1208021,2024-08-16 19:00:00+00:00,Old Trafford,R. Jones,Manchester United,Fulham,1,0,0,0,...,1,45,3,0,4,384,306,80,0.44,1.07
1,1208022,2024-08-17 11:30:00+00:00,Portman Road,T. Robinson,Ipswich,Liverpool,0,2,0,0,...,0,62,1,0,2,570,492,86,2.65,0.24
2,1208025,2024-08-17 14:00:00+00:00,St. James' Park,C. Pawson,Newcastle,Southampton,1,0,1,0,...,2,78,4,0,0,649,579,89,1.77,-0.37
3,1208023,2024-08-17 14:00:00+00:00,Emirates Stadium,J. Gillett,Arsenal,Wolves,2,0,1,0,...,1,47,2,0,4,375,307,82,0.47,-0.30
4,1208024,2024-08-17 14:00:00+00:00,Goodison Park,S. Hooper,Everton,Brighton,0,3,0,1,...,1,62,1,0,1,575,492,86,1.43,0.05


#### Đồng nhất thLực thể

In [8]:
# 1. Tải lại 2 bảng dữ liệu đã làm sạch
df_api = pd.read_csv("data_epl/api_football_clean.csv")
df_fd = pd.read_csv("data_epl/football_data_clean.csv")

# 2. Tạo cuốn từ điển đồng nhất tên đội bóng
# (Key: Tên của football-data_epl | Value: Tên chuẩn của API-Football)
team_mapping = {
    "Man United": "Manchester United",
    "Man City": "Manchester City",
    "Ipswich Town": "Ipswich",
    "Brighton Hove": "Brighton",
    "Wolverhampton": "Wolves",
    "Leicester City": "Leicester",
    "Nottingham": "Nottingham Forest", # Có thể thay đổi tuỳ theo tên thực tế API-Football lưu
    "Spurs": "Tottenham" # Phòng trường hợp có tên lóng
}

# 3. Tạo cột tên "đã phiên dịch" cho bảng football-data_epl
# Dùng hàm replace() để thay thế, nếu tên không có trong từ điển thì giữ nguyên
df_fd['home_team_mapped'] = df_fd['home_team'].replace(team_mapping)
df_fd['away_team_mapped'] = df_fd['away_team'].replace(team_mapping)

print("✅ Đã đồng nhất tên đội bóng xong!")

# 4. THỰC HIỆN CÚ NỐI BẢNG LỊCH SỬ (INNER JOIN)
# Ghép 2 bảng lại dựa trên tổ hợp Đội Nhà và Đội Khách
df_master = pd.merge(
    df_api,
    df_fd,
    left_on=['home_team', 'away_team'],           # Cột chuẩn của API-Football
    right_on=['home_team_mapped', 'away_team_mapped'], # Cột đã phiên dịch của Football-Data
    how='inner',
    suffixes=('_api', '_fd') # Nếu có cột trùng tên (như home_score), Pandas sẽ tự thêm hậu tố này để phân biệt
)

# 5. Nghiệm thu
print(f"\n--- BẢNG MASTER DATA HOÀN CHỈNH ---")
print(f"Tổng số trận ghép thành công: {df_master.shape[0]}/380")
print(f"Tổng số cột dữ liệu: {df_master.shape[1]}")

# Kiểm tra thử những cột quan trọng nhất của 1 trận
df_master[['match_date_api', 'home_team_api', 'away_team_api', 'home_score_api', 'referee_fd', 'home_expected_goals']].head()

✅ Đã đồng nhất tên đội bóng xong!

--- BẢNG MASTER DATA HOÀN CHỈNH ---
Tổng số trận ghép thành công: 380/380
Tổng số cột dữ liệu: 55


,match_date_api,home_team_api,away_team_api,home_score_api,referee_fd,home_expected_goals
0,2024-08-16 19:00:00+00:00,Manchester United,Fulham,1,Robert Jones,2.43
1,2024-08-17 11:30:00+00:00,Ipswich,Liverpool,0,Tim Robinson,0.45
2,2024-08-17 14:00:00+00:00,Newcastle,Southampton,1,Craig Pawson,0.25
3,2024-08-17 14:00:00+00:00,Arsenal,Wolves,2,Jarred Gillett,1.24
4,2024-08-17 14:00:00+00:00,Everton,Brighton,0,Simon Hooper,0.45


#### Làm sạch dữ liệu

In [10]:
# 1. Lên danh sách các cột thừa thãi cần vứt bỏ
cols_to_drop = [
    'home_team_mapped', 'away_team_mapped', # Cột từ điển tạm
    'home_team_fd', 'away_team_fd',         # Tên đội bóng bị trùng của nhánh 2
    'home_score_fd', 'away_score_fd',       # Tỷ số bị trùng
    'match_date_fd', 'data_source_fd', 'data_source_api', 'fd_match_id', # Các cột rác hệ thống
    'referee_api'                           # Vứt bỏ cột trọng tài bị trùng của API-Football
]

# Xóa các cột thừa
df_final = df_master.drop(columns=cols_to_drop, errors='ignore')

# 2. Đổi tên cột cho đẹp (Tự động xóa đuôi _api ở tất cả các cột)
rename_dict = {col: col.replace('_api', '') for col in df_final.columns if '_api' in col}

# Đổi riêng tên cột trọng tài của Football-Data cho chuẩn
rename_dict['referee_fd'] = 'referee'

df_final = df_final.rename(columns=rename_dict)

# Đảm bảo cột league_name nằm ở đầu cho đẹp mắt (nếu nó tồn tại)
if 'league_name' in df_final.columns:
    cols = ['match_id', 'league_name', 'match_date'] + [c for c in df_final.columns if c not in ['match_id', 'league_name', 'match_date']]
    df_final = df_final[cols]

# 3. Xuất file Master cuối cùng ra ổ cứng
final_output = "data_epl/master_matches_dataset.csv"
df_final.to_csv(final_output, index=False, encoding="utf-8-sig")

print(f"Bảng dữ liệu cuối cùng đã được tinh gọn còn {df_final.shape[1]} cột.")
print(f"Lưu tại: {final_output}")

# Xem thử giao diện cuối cùng
df_final.head()

Bảng dữ liệu cuối cùng đã được tinh gọn còn 44 cột.
Lưu tại: data_epl/master_matches_dataset.csv


,match_id,league_name,match_date,home_team,away_team,home_score,away_score,home_shots_on_goal,home_shots_off_goal,home_total_shots,...,away_ball_possession,away_yellow_cards,away_red_cards,away_goalkeeper_saves,away_total_passes,away_passes_accurate,away_passes_%,away_expected_goals,away_goals_prevented,referee
0,1208021,Premier League,2024-08-16 19:00:00+00:00,Manchester United,Fulham,1,0,5,7,14,...,45,3,0,4,384,306,80,0.44,1.07,Robert Jones
1,1208022,Premier League,2024-08-17 11:30:00+00:00,Ipswich,Liverpool,0,2,2,2,7,...,62,1,0,2,570,492,86,2.65,0.24,Tim Robinson
2,1208025,Premier League,2024-08-17 14:00:00+00:00,Newcastle,Southampton,1,0,1,0,3,...,78,4,0,0,649,579,89,1.77,-0.37,Craig Pawson
3,1208023,Premier League,2024-08-17 14:00:00+00:00,Arsenal,Wolves,2,0,6,6,18,...,47,2,0,4,375,307,82,0.47,-0.30,Jarred Gillett
4,1208024,Premier League,2024-08-17 14:00:00+00:00,Everton,Brighton,0,3,1,4,9,...,62,1,0,1,575,492,86,1.43,0.05,Simon Hooper


#### Tách các dòng data sang folder đúng

In [1]:
import json
import os

# Đảm bảo thư mục data_laliga tồn tại
os.makedirs("data_laliga", exist_ok=True)

# 1. Đọc file chứa dữ liệu bị lẫn lộn
with open("data_epl/api_football_statistics.json", "r", encoding="utf-8") as f:
    mixed_stats = json.load(f)

# 2. Đọc danh sách 380 trận Ngoại Hạng Anh gốc để lấy ID "chính chủ"
with open("data_epl/api_football_raw.json", "r", encoding="utf-8") as f:
    epl_raw = json.load(f)
    epl_ids = [m['fixture']['id'] for m in epl_raw.get("response", [])]

# 3. Tiến hành phân loại
epl_stats = []
laliga_stats = []

for stat in mixed_stats:
    if stat["fixture_id"] in epl_ids:
        epl_stats.append(stat)
    else:
        laliga_stats.append(stat)

# 4. Ghi trả lại đúng vị trí
with open("data_epl/api_football_statistics.json", "w", encoding="utf-8") as f:
    json.dump(epl_stats, f, ensure_ascii=False, indent=4)

with open("data_laliga/api_football_statistics.json", "w", encoding="utf-8") as f:
    json.dump(laliga_stats, f, ensure_ascii=False, indent=4)

print(f"Giữ lại {len(epl_stats)} trận cho EPL và chuyển {len(laliga_stats)} trận sang La Liga.")

Giữ lại 380 trận cho EPL và chuyển 95 trận sang La Liga.
